# BMI Category Classification with Logistic Regression

An educational supervised-learning workflow that predicts BMI categories from a tabular dataset using scikit-learn.

**Target:** `BmiClass` (Underweight, Normal weight, Overweight, Obese). This notebook does not predict diabetes and is not a medical tool.

## Dataset

Upload a CSV or ZIP containing a CSV with a `BmiClass` target column. The dataset is not stored in this repository. The notebook deliberately stops if that target column is missing rather than silently choosing an unrelated column.


In [ ]:
from google.colab import files
import io
import zipfile
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

uploaded = files.upload()
if not uploaded:
    raise ValueError("Upload a CSV or ZIP dataset to continue.")

file_name, file_bytes = next(iter(uploaded.items()))
if file_name.lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(file_bytes)) as archive:
        csv_files = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if not csv_files:
            raise ValueError("The ZIP archive does not contain a CSV file.")
        with archive.open(csv_files[0]) as csv_file:
            data = pd.read_csv(csv_file)
else:
    data = pd.read_csv(io.BytesIO(file_bytes))

TARGET = "BmiClass"
if TARGET not in data.columns:
    raise ValueError(f"Required target column '{TARGET}' was not found. Available columns: {list(data.columns)}")

data = data.dropna(subset=[TARGET]).copy()
X = data.drop(columns=[TARGET])
y = data[TARGET].astype(str)
if y.nunique() < 2:
    raise ValueError("The target must contain at least two distinct BMI categories.")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"Rows used: {len(data):,}")
print(f"Number of features: {X.shape[1]}")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nClass-wise evaluation:")
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, xticks_rotation=45)
plt.title("BMI Category Classification")
plt.tight_layout()
plt.show()


## Methodology notes

- Numeric values are imputed and standardised; categorical values are imputed and one-hot encoded.
- Preprocessing is fitted inside a pipeline using training data only, reducing data leakage.
- Stratified splitting preserves class proportions where possible.
- Class-wise precision, recall, F1-score, and the confusion matrix are reported alongside accuracy.
- `class_weight="balanced"` is included as a baseline option; compare it with other settings before drawing conclusions.

## Responsible use

BMI categories are not a diagnosis of diabetes or any other condition. This notebook is for learning machine-learning workflow design only. Do not use its predictions for clinical decisions.
